In [1]:
# Auto-generated by cram CLI — do not hand-edit.
# Rebuild with: python tools/cram.py build

import sys
from pathlib import Path

# Portably discover repo root containing 'engine' and add to sys.path
_cur = Path.cwd().resolve()
while _cur != _cur.parent:
    if (_cur / 'engine' / '__init__.py').is_file():
        if str(_cur) not in sys.path:
            sys.path.insert(0, str(_cur))
        break
    _cur = _cur.parent

from engine import Worksheet, apply_default_style

apply_default_style()

ws = Worksheet.load('lesson_22_ising_model_and_spin_kinetics.yaml')
ws.start()


$\def\AA{\text{Å}}$

# Lesson 22: The Ising Model: Exact Solutions, Phase Transitions, Spin Kinetics & Coarsening

_Session 9c44d588 · Estimated time: ~120 minutes · 42 problems_

**Recording your answers.** Submit each problem as you go. Your answers are saved to the workspace as you submit, *and* the running record at the bottom of the notebook keeps your progress. When you finish, run the last two cells — **`show_record()`** and **`finalize()`** — to record your participation. **If you reopened this workspace** (or the kernel restarted), use **Kernel → Restart Kernel and Run All Cells** before finalizing, so every answer is reloaded.

### Learning Objectives
- Define the Ising Hamiltonian, understand the discrete $\mathbb{Z}_2$ inversion symmetry, and map spins to lattice gases and binary alloys.
- Solve the 1D Ising model using the transfer matrix method and explain why continuous 1D systems with short-range interactions cannot exhibit a finite-temperature phase transition.
- Analyze Onsager's exact 2D solution, calculate the critical temperature $k_B T_c / J = 2 / \ln(1+\sqrt{2}) \approx 2.269$, and identify 2D critical exponents ($\alpha=0, \beta=1/8, \gamma=7/4, \nu=1$).
- Examine the finite-size magnetization probability distribution $P_L(M)$ and quantify the interfacial free energy barrier $\Delta F = 2\sigma L^{d-1}$ separating broken-symmetry ground states.
- Differentiate Glauber single-spin-flip dynamics (non-conserved order parameter, Model A, $L(t) \sim t^{1/2}$) from Kawasaki spin-exchange dynamics (conserved order parameter, Model B, $L(t) \sim t^{1/3}$).
- Explain critical slowing down ($\tau \sim \xi^z$) and implement the Swendsen-Wang and Wolff cluster algorithms based on the Fortuin-Kasteleyn random cluster mapping ($p = 1 - e^{-2\beta J}$).


In [2]:
ws.render_section('sec_ising_hamiltonian_and_mappings')

### The Ising Hamiltonian

Proposed by Wilhelm Lenz (1920) and solved in 1D by Ernst Ising (1925), the Ising model represents discrete magnetic dipoles on a $d$-dimensional lattice. Each site $i \in \{1, \dots, N\}$ carries a scalar spin variable $s_i \in \{-1, +1\}$. The Hamiltonian is:

$$\mathcal{H}(\{s_i\}) = -J \sum_{\langle i, j \rangle} s_i s_j - h \sum_{i=1}^N s_i$$

where $J > 0$ favors ferromagnetic alignment, $\langle i, j \rangle$ denotes distinct nearest-neighbor pairs, and $h$ is an external magnetic field. When $h = 0$, the Hamiltonian possesses a global discrete $\mathbb{Z}_2$ **inversion symmetry**: flipping all spins $s_i \to -s_i$ leaves $\mathcal{H}$ invariant.

### The Lattice Gas Mapping (Liquid-Gas Coexistence)

The Ising model maps isomorphically to a lattice gas of fluid molecules. Let $n_i \in \{0, 1\}$ denote the occupation number of site $i$ ($n_i = 1$ if occupied, $0$ if empty). Relating the spin to occupation via:

$$s_i = 2n_i - 1 \iff n_i = \frac{1 + s_i}{2}$$

The lattice gas grand canonical Hamiltonian with nearest-neighbor attraction $-\epsilon$ and chemical potential $\mu$ is:

$$\mathcal{H}_{\text{LG}} = -\epsilon \sum_{\langle i, j \rangle} n_i n_j - \mu \sum_i n_i$$

Substituting $n_i = (s_i + 1)/2$ maps this directly to the Ising Hamiltonian with:

$$J = \frac{\epsilon}{4}, \qquad h = \frac{\mu - \mu_0}{2}$$

where $\mu_0 = -z\epsilon / 2$ is the chemical potential at liquid-gas coexistence ($z$ being the lattice coordination number). The spontaneous magnetization $M = \frac{1}{N}\sum s_i$ corresponds to the density difference between liquid and vapor phases:

$$\rho - \rho_c = \frac{1}{2} M$$

### Binary Alloy Mapping (Order-Disorder Transition)

In a binary alloy (e.g. Cu-Au or Fe-Al), let $s_i = +1$ denote an atom of species $A$ and $s_i = -1$ denote species $B$. Pair interactions $V_{AA}, V_{BB}, V_{AB}$ map onto the Ising model with effective exchange parameter:

$$J = \frac{1}{4}(2 V_{AB} - V_{AA} - V_{BB})$$

If $J > 0$, unlike neighbors are favored (ordering alloy / antiferromagnet); if $J < 0$, like neighbors are favored (phase-separating mixture).


In [3]:
ws.render_section('sec_exact_solutions_and_onsager')

### The 1D Ising Chain and the Transfer Matrix

For a 1D chain of $N$ spins with periodic boundary conditions ($s_{N+1} = s_1$) in field $h$, the partition function is:

$$Z = \sum_{\{s_i\}} \prod_{i=1}^N \exp\left[ \beta J s_i s_{i+1} + \frac{1}{2}\beta h (s_i + s_{i+1}) \right] = \text{Tr}\left( \mathbf{T}^N \right)$$

where $\mathbf{T}$ is the $2 \times 2$ **transfer matrix**:

$$\mathbf{T} = \begin{pmatrix} e^{\beta(J + h)} & e^{-\beta J} \\ e^{-\beta J} & e^{\beta(J - h)} \end{pmatrix}$$

The eigenvalues of $\mathbf{T}$ for $h = 0$ are $\lambda_1 = 2\cosh(\beta J)$ and $\lambda_2 = 2\sinh(\beta J)$. In the thermodynamic limit $N \to \infty$, the partition function is dominated by the largest eigenvalue:

$$Z = \lambda_1^N + \lambda_2^N = \lambda_1^N \left[ 1 + \left( \frac{\lambda_2}{\lambda_1} \right)^N \right] \approx \lambda_1^N = (2\cosh\beta J)^N$$

The spin-spin correlation function is:

$$\langle s_i s_{i+r} \rangle = \left( \frac{\lambda_2}{\lambda_1} \right)^r = (\tanh\beta J)^r = e^{-r / \xi}$$

where the correlation length is $\xi = -1 / \ln(\tanh\beta J) \approx \frac{1}{2} e^{2\beta J}$ as $T \to 0$. Because $\lambda_1 > \lambda_2 > 0$ for all $T > 0$, the free energy is strictly analytic: **there is no phase transition at any finite temperature in 1D!**

### Onsager's Exact 2D Solution (1944)

Lars Onsager solved the zero-field 2D square lattice Ising model exactly. The critical coupling satisfies:

$$\sinh(2\beta_c J) = 1 \implies \beta_c J = \frac{1}{2} \ln(1 + \sqrt{2}) \approx 0.440687$$

which yields the exact critical temperature:

$$\frac{k_B T_c}{J} = \frac{2}{\ln(1 + \sqrt{2})} \approx 2.269185$$

C. N. Yang (1952) subsequently derived the exact spontaneous magnetization below $T_c$:

$$M_0(T) = \left[ 1 - \frac{1}{[\sinh(2\beta J)]^4} \right]^{1/8} \quad (T < T_c)$$

Near $T_c$, $M_0(T) \propto (T_c - T)^{\beta}$ with critical exponent $\beta = 1/8 = 0.125$ (distinct from mean-field $\beta_{\text{MF}} = 1/2$).


In [4]:
ws.render_section('sec_magnetization_distribution_and_interfaces')

### The Order Parameter Distribution $P_L(M)$

In a finite lattice of linear dimension $L$ ($N = L^d$ spins), the total magnetization per spin is $M = \frac{1}{N}\sum_{i=1}^N s_i$. The probability density $P_L(M)$ reveals the underlying Landau effective free energy $\mathcal{F}_L(M) = -k_B T \ln P_L(M)$:

1. **High-Temperature Disordered Phase ($T > T_c$):** Spins are largely uncorrelated ($\xi \ll L$). By the Central Limit Theorem, $P_L(M)$ is a single Gaussian centered at $M = 0$:

$$P_L(M) \propto \exp\left( -\frac{N M^2}{2 k_B T \chi_T} \right)$$

2. **Critical Point ($T = T_c$):** Fluctuations occur on all length scales up to $L$ ($\xi \sim L$). The Gaussian term vanishes, and $P_L(M)$ flattens into a non-Gaussian distribution governed by the quartic term:

$$P_L(M) \propto \exp\left( -u L^{d - 4\beta/\nu} M^4 \right)$$

3. **Low-Temperature Ordered Phase ($T < T_c$):** Spontaneous symmetry breaking splits $P_L(M)$ into **two symmetric peaks** at $M \approx \pm M_0(T)$. Configurations near $M = 0$ must contain two system-spanning domain walls separating regions of spin up and spin down.

### Interfacial Free Energy Barrier

The free energy penalty to create two flat domain walls of cross-sectional area $L^{d-1}$ is:

$$\Delta \mathcal{F}_L = 2 \sigma(T) L^{d-1}$$

where $\sigma(T)$ is the surface tension between $+M_0$ and $-M_0$ phases. Consequently, the probability of visiting $M = 0$ is suppressed exponentially:

$$\frac{P_L(M = 0)}{P_L(M = M_0)} \approx \exp\left( -2\beta \sigma(T) L^{d-1} \right)$$

In Monte Carlo simulations with local single-spin flips, tunneling between $+M_0$ and $-M_0$ requires an astronomically long Kramers time $\tau_{\text{tunnel}} \sim \exp(2\beta \sigma L^{d-1})$, trapping the system in a single broken-symmetry state.


In [5]:
ws.render_section('concept_ising_magnetization_pdf')

### Interactive Concept: Magnetization Distribution P_L(M) and Barrier

Explore the magnetization probability distribution $P_L(M)$ and Landau free energy profile $\mathcal{F}(M) = -\ln P_L(M)$.
Adjust temperature ratio $T/T_c$, lattice size $L$, and external magnetic field $h$ to observe bimodal splitting and field tilt.

**Key Insight:** Below $T_c$, $P_L(M)$ splits into two sharp peaks at $\pm M_0$. As $L$ increases, the barrier at $M = 0$ scales as $2\sigma L$, exponentially suppressing transitions between ground states. Adding a magnetic field $h > 0$ tilts the potential, exponentially stabilizing the $+M_0$ phase.

In [6]:
ws.render_section('q22_01')

**Q1 · easy** · 💡

What discrete global symmetry does the zero-field ($h = 0$) Ising Hamiltonian possess?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Consider what happens to the product $s_i s_j$ when every spin changes sign: $(-s_i)(-s_j) = s_i s_j$.

</div>
</details>

- [ ] **(A)** Continuous rotational symmetry $O(3)$
- [ ] **(B)** Discrete spin-inversion symmetry $\mathbb{Z}_2$: invariance under $s_i \to -s_i$ for all $i$
- [ ] **(C)** Gauge symmetry $U(1)$
- [ ] **(D)** Lorentz invariance

In [7]:
ws.render_section('q22_02')

**Q2 · medium** · 💡

Under the mapping from the Ising model to the lattice gas ($s_i = 2n_i - 1$), what physical quantity in the fluid corresponds to the Ising magnetization $M = \frac{1}{N}\sum s_i$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Relate $\rho = \frac{1}{N}\sum n_i$ to $M = \frac{1}{N}\sum s_i$.

</div>
</details>

- [ ] **(A)** Total kinetic energy of the fluid
- [ ] **(B)** Deviation of fluid density from the critical density: $\rho - \rho_c = \frac{1}{2} M$
- [ ] **(C)** The viscosity of the fluid
- [ ] **(D)** The temperature of the fluid

In [8]:
ws.render_section('q22_03')

**Q3 · easy** · 💡

Using Onsager's exact formula $\frac{k_B T_c}{J} = \frac{2}{\ln(1 + \sqrt{2})}$, calculate the critical temperature $k_B T_c / J$ for the 2D square lattice Ising model to 3 decimal places.

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Evaluate $2 / \ln(1 + 1.414214)$.

</div>
</details>

_Numeric answer required._

In [9]:
ws.render_section('q22_04')

**Q4 · medium** · 💡

Why cannot a 1D Ising chain with finite-range interactions undergo a spontaneous phase transition at any temperature $T > 0$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Think about the free energy $\Delta F = \Delta E - T \Delta S$ to insert a single domain boundary in a chain of $N$ sites.

</div>
</details>

- [ ] **(A)** Because creating a domain wall (kink) costs a finite energy $\Delta E = 2J$, but introduces an entropic contribution $-T \Delta S = -k_B T \ln N$ that diverges logarithmically as $N \to \infty$, making domain walls spontaneously populate the chain.
- [ ] **(B)** Because the transfer matrix has zero eigenvalues.
- [ ] **(C)** Because spins cannot be flipped in one dimension.
- [ ] **(D)** Because the 1D lattice has no nearest neighbors.

In [10]:
ws.render_section('q22_05')

**Q5 · hard** · 💡

What is the critical exponent $\beta$ governing spontaneous magnetization $M_0 \propto (T_c - T)^{\beta}$ in the 2D Ising model according to Yang's exact solution, and how does it compare to Mean-Field Theory?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall Yang's formula $M_0 = [1 - \sinh^{-4}(2\beta J)]^{1/8}$.

</div>
</details>

- [ ] **(A)** $\beta = 1/8$ in 2D Onsager, whereas Mean-Field Theory predicts $\beta_{\text{MF}} = 1/2$.
- [ ] **(B)** $\beta = 1/2$ in both 2D and Mean-Field.
- [ ] **(C)** $\beta = 0$ in 2D, meaning magnetization jumps discontinuously.
- [ ] **(D)** $\beta = 1$ in 2D.

In [11]:
ws.render_section('q22_06')

**Q6 · medium** · 💡

In the concept cell, at $T/T_c = 0.85$ and $h = 0$, what is the qualitative shape of the probability density function $P_L(M)$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Look at the plot in the concept cell for $T < T_c$.

</div>
</details>

- [ ] **(A)** A single narrow delta peak centered at $M = 0$
- [ ] **(B)** A symmetric bimodal distribution with two distinct peaks at $M \approx +M_0$ and $M \approx -M_0$, separated by a deep valley at $M = 0$
- [ ] **(C)** A uniform flat line from $M = -1$ to $M = +1$
- [ ] **(D)** An exponentially decaying curve starting at $M = -1$

In [12]:
ws.render_section('q22_07')

**Q7 · medium** · 💡

In a 2D square lattice of size $L = 32$, two straight domain walls span the periodic box. If each wall has length $L = 32$ and surface tension $\beta \sigma = 0.15$ per unit length, calculate the total dimensionless interfacial free energy penalty $\Delta \mathcal{F} / k_B T = 2 \beta \sigma L$.

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Multiply $2 \times 0.15 \times 32$.

</div>
</details>

_Numeric answer required._

In [13]:
ws.render_section('q22_08')

**Q8 · medium** · 💡

Why do periodic boundary conditions require a minimum of TWO domain walls (rather than one) to transition from a fully up-spin state to a fully down-spin state?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Draw a closed loop on a torus where spins go $+1 \to -1 \to +1$.

</div>
</details>

- [ ] **(A)** Because spins can only flip in pairs.
- [ ] **(B)** Because periodic boundaries identify $x = 0$ with $x = L$; a slice of down-spins embedded in an up-spin bulk necessarily has an entry boundary ($+ \to -$) and an exit boundary ($- \to +$).
- [ ] **(C)** Because energy must be conserved.
- [ ] **(D)** Because the magnetic field is always zero.

In [14]:
ws.render_section('q22_09')

**Q9 · easy** · 💡

When a positive external magnetic field $h > 0$ is applied in the concept cell, how does the magnetization distribution $P_L(M)$ respond?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall the Zeeman energy term $-h \sum s_i = -N h M$.

</div>
</details>

- [ ] **(A)** The two peaks remain exactly symmetric.
- [ ] **(B)** The peak at $+M_0$ grows substantially while the peak at $-M_0$ is exponentially suppressed by the Zeeman energy factor $e^{2\beta N h M_0}$.
- [ ] **(C)** Both peaks shift immediately to $M = 0$.
- [ ] **(D)** The temperature of the system drops to zero.

In [15]:
ws.render_section('q22_10')

**Q10 · medium** · 💡

Which of the following critical exponents characterize the 2D Ising universality class? (Select all that apply)

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall the exact 2D Ising exponents $(\alpha, \beta, \gamma, \nu, \eta) = (0, 1/8, 7/4, 1, 1/4)$.

</div>
</details>

- [ ] **(A)** Specific heat exponent $\alpha = 0$ (logarithmic divergence $C_V \propto \ln|T - T_c|$)
- [ ] **(B)** Order parameter exponent $\beta = 1/8$
- [ ] **(C)** Magnetic susceptibility exponent $\gamma = 7/4$
- [ ] **(D)** Correlation length exponent $\nu = 1$
- [ ] **(E)** Mean-field exponent $\gamma_{\text{MF}} = 1$

In [16]:
ws.render_section('sec_glauber_vs_kawasaki_kinetics')

### Non-Equilibrium Quenches and Phase Separation

When an Ising system is rapidly quenched from a high temperature $T \gg T_c$ to below $T_c$, the uniform disordered state becomes thermodynamically unstable. Microscopic domains of up-spins and down-spins nucleate and grow through **coarsening** (domain growth).

The physical mechanism and rate of coarsening depend critically on whether the order parameter is conserved:

### Glauber Dynamics (Model A: Non-Conserved Order Parameter)

- **Elementary Move:** A single spin $s_i$ is chosen at random and flipped: $s_i \to -s_i$.
- **Conservation:** The total magnetization $M = \frac{1}{N}\sum s_i$ is **not conserved** ($\dot{M} \ne 0$).
- **Physics:** Models magnetic materials where domain walls move freely under external and internal magnetic fields.
- **Coarsening Law (Lifshitz-Allen-Cahn):** Domain wall motion is driven by local interface curvature $\kappa$:

$$v_n \propto -\kappa$$

Since curvature scales as $\kappa \sim 1/L(t)$ and velocity is $dL/dt$, we have $dL/dt \sim 1/L$, yielding the parabolic growth law:

$$L(t) \propto t^{1/2}$$

### Kawasaki Dynamics (Model B: Conserved Order Parameter)

- **Elementary Move:** Two nearest-neighbor spins with opposite signs ($s_i = +1, s_j = -1$) are **exchanged**: $s_i \leftrightarrow s_j$.
- **Conservation:** The total magnetization $M = \frac{1}{N}\sum s_i$ is **strictly conserved** at its initial value.
- **Physics:** Models binary alloys ($A-B$ segregation) and liquid-gas phase separation, where particle numbers of each species are fixed.
- **Coarsening Law (Lifshitz-Slyozov-Wagner / LSW):** Domain growth proceeds via **Ostwald ripening** (small domains evaporate and diffuse through the matrix to deposit onto larger domains) and interfacial diffusion:

$$L(t) \propto t^{1/3}$$

Because mass must diffuse across domains, growth under Kawasaki dynamics is significantly slower ($t^{1/3}$) than curvature-driven Glauber growth ($t^{1/2}$).


In [17]:
ws.render_section('concept_glauber_vs_kawasaki_coarsening')

### Interactive Concept: Domain Coarsening (Glauber t^(1/2) vs. Kawasaki t^(1/3))

Compare non-conserved Glauber dynamics with conserved Kawasaki dynamics following a deep quench to $T = 0.5 T_c$.
Switch between the two dynamical modes, adjust quench time $t$, and observe domain structure and characteristic domain size $L(t)$.

**Key Insight:** Glauber dynamics rapidly collapses isolated minority clusters through curvature-driven shrinkage ($L(t) \sim t^{1/2}$), rapidly magnetizing the entire lattice. Kawasaki dynamics maintains fixed total composition, forcing the system to form interconnected bicontinuous structures (spinodal decomposition) that coarsen via slow diffusive ripening ($L(t) \sim t^{1/3}$).

In [18]:
ws.render_section('q22_11')

**Q11 · easy** · 💡

What is the defining difference between Glauber dynamics and Kawasaki dynamics?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall the elementary trial move in each kinetic model.

</div>
</details>

- [ ] **(A)** Glauber dynamics uses molecular dynamics, whereas Kawasaki dynamics uses Monte Carlo.
- [ ] **(B)** Glauber flips single spins (non-conserved magnetization), whereas Kawasaki exchanges pairs of opposite spins (strictly conserved magnetization).
- [ ] **(C)** Kawasaki dynamics only operates in one dimension.
- [ ] **(D)** Glauber dynamics requires quantum mechanical wavefunctions.

In [19]:
ws.render_section('q22_12')

**Q12 · medium** · 💡

What is the characteristic domain growth law $L(t)$ for non-conserved (Glauber / Model A) domain coarsening in an Ising model?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Domain wall velocity is proportional to local curvature: $dL/dt \sim 1/L$.

</div>
</details>

- [ ] **(A)** $L(t) \propto \ln t$
- [ ] **(B)** $L(t) \propto t^{1/2}$ (Lifshitz-Allen-Cahn law)
- [ ] **(C)** $L(t) \propto t^{1/3}$ (Lifshitz-Slyozov law)
- [ ] **(D)** $L(t) \propto t^2$

In [20]:
ws.render_section('q22_13')

**Q13 · medium** · 💡

What is the characteristic domain growth law $L(t)$ for conserved (Kawasaki / Model B) phase separation during spinodal decomposition?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Domain growth is limited by bulk and interfacial diffusion (Ostwald ripening): $dL/dt \sim D \sigma / (k_B T L^2)$.

</div>
</details>

- [ ] **(A)** $L(t) \propto t^{1/3}$ (Lifshitz-Slyozov-Wagner law)
- [ ] **(B)** $L(t) \propto t^{1/2}$
- [ ] **(C)** $L(t) \propto e^{t}$
- [ ] **(D)** $L(t)$ remains constant forever

In [21]:
ws.render_section('q22_14')

**Q14 · medium** · 💡

Under Kawasaki coarsening ($L(t) \propto t^{1/3}$), if the average domain size is $L_1 = 5.0\text{ nm}$ at $t = 100\text{ MCS}$, what is the expected domain size $L_2$ in nm at $t = 800\text{ MCS}$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Calculate $L_2 = L_1 \times (800 / 100)^{1/3}$.

</div>
</details>

_Numeric answer required (in nm)._

In [22]:
ws.render_section('q22_15')

**Q15 · medium** · 💡

For the same initial domain size $L_1 = 5.0\text{ nm}$ at $t = 100\text{ MCS}$, what would the domain size be in nm at $t = 400\text{ MCS}$ under Glauber coarsening ($L(t) \propto t^{1/2}$)?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Calculate $L_2 = L_1 \times (400 / 100)^{1/2}$.

</div>
</details>

_Numeric answer required (in nm)._

In [23]:
ws.render_section('q22_16')

**Q16 · medium** · 💡

In the concept cell, why does Kawasaki dynamics with initial magnetization $M_0 = 0.15$ form an interconnected (bicontinuous) domain labyrinth rather than quickly reaching a single uniform phase?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Remember that Kawasaki moves cannot change $\sum s_i$.

</div>
</details>

- [ ] **(A)** Because the code prevents spins from flipping at the edges.
- [ ] **(B)** Because the global fraction of up and down spins is fixed; neither phase can consume the other, forcing the system to maintain both phases separated by domain boundaries.
- [ ] **(C)** Because the temperature is set to infinity.
- [ ] **(D)** Because Kawasaki dynamics is strictly non-ergodic.

In [24]:
ws.render_section('q22_17')

**Q17 · hard** · 💡

In spinodal decomposition, what is the 'spinodal line' in the phase diagram?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall the distinction between nucleation (metastable regime) and spinodal decomposition (unstable regime $\partial^2 f / \partial c^2 < 0$).

</div>
</details>

- [ ] **(A)** The boundary where the system melts into a liquid.
- [ ] **(B)** The curve where the thermodynamic stability limit is reached ($\partial^2 f / \partial M^2 = 0$), inside which the uniform state is unstable to infinitesimal fluctuations with no nucleation barrier.
- [ ] **(C)** The line where the magnetic field equals 1 Tesla.
- [ ] **(D)** The boundary where Monte Carlo simulations freeze.

In [25]:
ws.render_section('q22_18')

**Q18 · medium** · 💡

Which of the following physical systems are modeled by Kawasaki (Model B) conserved spin dynamics? (Select all that apply)

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Identify processes where the total number of particles or molecules of each species is strictly conserved.

</div>
</details>

- [ ] **(A)** Phase separation of a binary alloy (e.g. Al-Li or Fe-Cr spinodal decomposition)
- [ ] **(B)** Demixing of two immiscible polymer blends
- [ ] **(C)** Uniaxial ferromagnet domain reversal under a magnetic field
- [ ] **(D)** Liquid-gas phase separation in a sealed, constant-volume chamber

In [26]:
ws.render_section('q22_19')

**Q19 · medium** · 💡

In Glauber dynamics, what is the transition probability $W(s_i \to -s_i)$ originally proposed by Roy Glauber in 1963?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Notice that Glauber dynamics satisfies detailed balance with a smooth sigmoid transition rate.

</div>
</details>

- [ ] **(A)** $W(s_i \to -s_i) = \frac{1}{2}\left( 1 - s_i \tanh(\beta E_{\text{local}}) \right)$, where $E_{\text{local}} = J \sum_{j \in nn} s_j + h$
- [ ] **(B)** $W(s_i \to -s_i) = e^{-\beta \Delta E}$ unconditionally
- [ ] **(C)** $W(s_i \to -s_i) = 0.5$ always
- [ ] **(D)** $W(s_i \to -s_i) = \frac{1}{1 + s_i^2}$

In [27]:
ws.render_section('q22_20')

**Q20 · easy** · 💡

Why does Ostwald ripening occur in conserved phase separation?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall the Gibbs-Thomson effect relating droplet curvature to vapor pressure/solubility.

</div>
</details>

- [ ] **(A)** Small domains have higher surface-to-volume ratio and higher Gibbs-Thomson chemical potential, causing solute to evaporate from small droplets and condense onto larger droplets.
- [ ] **(B)** Small domains attract each other gravitationally.
- [ ] **(C)** Large domains spontaneously explode into small droplets.
- [ ] **(D)** The system temperature increases continuously.

In [28]:
ws.render_section('sec_critical_slowing_down_and_swendsen_wang')

### Critical Slowing Down

Near the critical temperature $T_c$, the correlation length diverges as $\xi \propto |T - T_c|^{-\nu}$. Fluctuations become correlated over enormous spatial domains. In local single-spin-flip algorithms (Metropolis or Glauber), propagating information across a domain of size $\xi$ requires a random walk of domain walls, causing the **autocorrelation time** $\tau$ to diverge:

$$\tau \propto \xi^z \propto |T - T_c|^{-z\nu}$$

where $z$ is the **dynamic critical exponent**. For local single-spin-flip Metropolis in 2D, $z \approx 2.17$. On a lattice of size $L \approx \xi$, the computational effort per independent sample scales as:

$$\text{CPU Time} \sim L^d \times \tau \sim L^{d + z} \approx L^{4.17}$$

Simulating $L = 1024$ near $T_c$ with local flips becomes completely intractable due to this **critical slowing down**.

### The Fortuin-Kasteleyn Mapping and Swendsen-Wang

In 1987, Robert Swendsen and Jian-Sheng Wang eliminated critical slowing down by mapping the Ising spin model to the **Fortuin-Kasteleyn random cluster model**. Instead of flipping single spins, whole correlated clusters are flipped simultaneously.

**Swendsen-Wang Algorithm Steps:**
1. **Bond Activation:** Iterate over all nearest-neighbor pairs $\langle i, j \rangle$. If the spins are parallel ($s_i = s_j$), activate a bond between them with probability:

$$p = 1 - e^{-2\beta J}$$

If the spins are antiparallel ($s_i \ne s_j$), no bond is formed ($p = 0$).
2. **Cluster Identification:** Find all connected components of sites linked by active bonds (using Hoshen-Kopelman or Disjoint-Set / Union-Find).
3. **Cluster Spin Flip:** For each connected cluster independently, choose a new spin orientation $s_{\text{cluster}} \in \{-1, +1\}$ with equal probability $1/2$ (or with field-dependent probability if $h \ne 0$). Assign this orientation to all spins in the cluster.

### Wolff Single-Cluster Algorithm

Ulli Wolff (1989) refined the technique: pick a random seed spin and recursively grow a single cluster by adding parallel neighbors with probability $p = 1 - e^{-2\beta J}$, then flip the cluster unconditionally. Cluster algorithms dramatically slash the dynamic exponent in 2D from $z \approx 2.17$ down to $z_{\text{SW}} \approx 0.35$, completely eradicating critical slowing down!


In [29]:
ws.render_section('concept_swendsen_wang_clusters')

### Interactive Concept: Swendsen-Wang Fortuin-Kasteleyn Clusters

Visualize the Fortuin-Kasteleyn bond creation and cluster formation across temperatures.
Adjust $T/T_c$ from $0.5$ (cold ordered) to $1.5$ (hot disordered) and observe the cluster size distribution and percolation at $T_c$.

**Key Insight:** At $T \gg T_c$, bonds rarely form ($p \approx 0$) and clusters are isolated single spins. At $T \ll T_c$, almost all parallel bonds activate ($p \approx 1$) and the entire lattice forms a single giant cluster. Exactly at $T = T_c$, a scale-invariant fractal cluster percolates across the system, enabling global configuration updates in a single Monte Carlo step!

In [30]:
ws.render_section('q22_21')

**Q21 · easy** · 💡

In the Swendsen-Wang cluster algorithm, what is the probability $p$ of placing an active bond between two adjacent parallel spins ($s_i = s_j$)?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall the Fortuin-Kasteleyn random cluster bond probability.

</div>
</details>

- [ ] **(A)** $p = 1 - e^{-2\beta J}$
- [ ] **(B)** $p = e^{-\beta J}$
- [ ] **(C)** $p = 0.5$
- [ ] **(D)** $p = \tanh(\beta J)$

In [31]:
ws.render_section('q22_22')

**Q22 · medium** · 💡

At the 2D Ising critical point where $\beta_c J = \frac{1}{2}\ln(1 + \sqrt{2}) \approx 0.440687$, calculate the exact bond probability $p_c = 1 - e^{-2\beta_c J}$ to 4 decimal places.

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Notice that $e^{2\beta_c J} = 1 + \sqrt{2}$. Evaluate $1 - \frac{1}{1 + \sqrt{2}}$.

</div>
</details>

_Numeric answer required._

In [32]:
ws.render_section('q22_23')

**Q23 · medium** · 💡

How does the Swendsen-Wang algorithm determine the new spin value for each identified cluster in zero magnetic field?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Review step 3 of the Swendsen-Wang algorithm.

</div>
</details>

- [ ] **(A)** Every cluster flips its spins to the opposite of its current value.
- [ ] **(B)** Each cluster is independently assigned $+1$ or $-1$ with equal probability $1/2$.
- [ ] **(C)** All clusters are set to $+1$.
- [ ] **(D)** The cluster spin is determined by a continuous Gaussian distribution.

In [33]:
ws.render_section('q22_24')

**Q24 · medium** · 💡

What happens to the dynamic critical exponent $z$ in 2D when switching from local single-spin Metropolis to the Swendsen-Wang cluster algorithm?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Cluster flips update large correlated regions in a single step, decoupling the autocorrelation time from diffusion.

</div>
</details>

- [ ] **(A)** $z$ increases from 2.17 to 4.0.
- [ ] **(B)** $z$ drops drastically from $z \approx 2.17$ down to $z \approx 0.35$, dramatically suppressing critical slowing down.
- [ ] **(C)** $z$ remains strictly unchanged at 2.0.
- [ ] **(D)** $z$ becomes negative.

In [34]:
ws.render_section('q22_25')

**Q25 · hard** · 💡

On an $L = 128$ lattice at $T_c$, single-spin Metropolis has autocorrelation time $\tau_{\text{local}} \propto L^{2.17}$. Swendsen-Wang has $\tau_{\text{SW}} \propto L^{0.35}$. What is the ratio $\tau_{\text{local}} / \tau_{\text{SW}} = L^{2.17 - 0.35} = L^{1.82}$ for $L = 128$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Calculate $128^{1.82}$.

</div>
</details>

_Numeric answer required._

In [35]:
ws.render_section('q22_26')

**Q26 · medium** · 💡

What key difference distinguishes the Wolff cluster algorithm from the Swendsen-Wang algorithm?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Why is Wolff called the 'single-cluster algorithm'?

</div>
</details>

- [ ] **(A)** Wolff grows and flips a SINGLE cluster around a randomly chosen seed site, rather than partitioning the entire lattice into clusters at every step.
- [ ] **(B)** Wolff algorithm only works in 1D.
- [ ] **(C)** Wolff algorithm does not satisfy detailed balance.
- [ ] **(D)** Wolff algorithm requires computing matrix inverses.

In [36]:
ws.render_section('q22_27')

**Q27 · medium** · 💡

In the concept cell, what happens to the Fortuin-Kasteleyn clusters at low temperature $T / T_c = 0.5$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Calculate $p = 1 - e^{-2(0.44/0.5)} = 1 - e^{-1.76} \approx 0.83$.

</div>
</details>

- [ ] **(A)** Every spin is isolated in its own private 1-site cluster.
- [ ] **(B)** A single giant cluster spans virtually the entire lattice because $p = 1 - e^{-2\beta J} \approx 0.97$, linking nearly all parallel spins together.
- [ ] **(C)** All bonds disappear.
- [ ] **(D)** The cluster size follows a Cauchy distribution.

In [37]:
ws.render_section('q22_28')

**Q28 · medium** · 💡

Under what conditions do Swendsen-Wang and Wolff cluster algorithms LOSE their immense efficiency advantage over local Metropolis? (Select all that apply)

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Consider how external fields and geometric frustration affect the Fortuin-Kasteleyn percolation mapping.

</div>
</details>

- [ ] **(A)** In the presence of a strong external magnetic field $h \ne 0$, where clusters grow to encompass the entire box and flips are rejected or field-dominated.
- [ ] **(B)** In frustrated spin systems like spin glasses or antiferromagnets on triangular lattices, where conflicting interactions prevent clean cluster formation.
- [ ] **(C)** In standard zero-field 2D ferromagnetic Ising models near $T_c$.
- [ ] **(D)** When calculating the microcanonical density of states with Wang-Landau.

In [38]:
ws.render_section('q22_29')

**Q29 · hard** · 💡

What data structure and algorithm is standardly used to identify connected components in Swendsen-Wang in linear time $\mathcal{O}(N)$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

How do percolation algorithms track which sites belong to which cluster?

</div>
</details>

- [ ] **(A)** Dijkstra's shortest path algorithm
- [ ] **(B)** Disjoint-Set (Union-Find) with path compression or the Hoshen-Kopelman algorithm
- [ ] **(C)** Fast Fourier Transform (FFT)
- [ ] **(D)** Singular Value Decomposition (SVD)

In [39]:
ws.render_section('q22_30')

**Q30 · easy** · 💡

At very high temperature $T \gg T_c$, what is the limiting behavior of the bond probability $p = 1 - e^{-2\beta J}$ in the Swendsen-Wang algorithm?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

As $T \to \infty$, $\beta = 1/k_B T \to 0$. What is $1 - e^0$?

</div>
</details>

- [ ] **(A)** $p \to 1$
- [ ] **(B)** $p \to 0$, so no bonds form and every spin flips independently with probability 1/2, recovering independent random sampling.
- [ ] **(C)** $p \to 0.5$
- [ ] **(D)** $p \to \infty$

In [40]:
ws.render_section('q22_31')

**Q31 · medium** · 💡

In a 1D Ising model at temperature where $\tanh(\beta J) = 0.90$, calculate the correlation length $\xi = -1 / \ln(\tanh\beta J)$ in lattice units to 2 decimal places.

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compute $-1 / \ln(0.90)$.

</div>
</details>

_Numeric answer required._

In [41]:
ws.render_section('q22_32')

**Q32 · hard** · 💡

What is the physical origin of the finite-size rounding of the second-order phase transition in a simulation of size $L$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Can an infinite correlation length $\xi = \infty$ exist inside a box of size $L$?

</div>
</details>

- [ ] **(A)** In finite systems, the correlation length cannot grow larger than the box size ($\xi \le L$), rounding the true mathematical singularities of the infinite-volume limit into smooth, finite peaks.
- [ ] **(B)** Floating-point errors truncate the partition function.
- [ ] **(C)** The number of spins is not divisible by 2.
- [ ] **(D)** The magnetic field diverges.

In [42]:
ws.render_section('q22_33')

**Q33 · medium** · 💡

For an antiferromagnetic Ising model ($J < 0$) on a bipartite lattice (e.g. square lattice) with $h = 0$, how can the model be mapped onto a ferromagnetic model ($J > 0$)?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

On a bipartite lattice, every neighbor of sublattice A is in sublattice B. If $s_i \to -s_i$ for $i \in B$, what happens to $s_i s_j$?

</div>
</details>

- [ ] **(A)** By applying an infinite magnetic field.
- [ ] **(B)** By performing a gauge transformation that flips the sign of all spins on one of the two sublattices: $s_i \to -s_i$ for $i \in B$.
- [ ] **(C)** By rotating the lattice by 45 degrees.
- [ ] **(D)** An antiferromagnet cannot be mapped to a ferromagnet.

In [43]:
ws.render_section('q22_34')

**Q34 · hard** · 💡

Why does an antiferromagnetic Ising model ($J < 0$) on a TRIANGULAR lattice fail to have a simple ordered ground state (exhibiting geometric frustration)?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Try assigning $\pm 1$ to three vertices of a triangle such that every edge has opposite spins.

</div>
</details>

- [ ] **(A)** Because triangles have 3 edges; it is impossible for all 3 pairs of spins on a triangle to be simultaneously antiparallel, leaving a non-zero residual entropy $S_0 > 0$ at $T = 0$ (Wannier 1950).
- [ ] **(B)** Because triangular lattices have coordination number zero.
- [ ] **(C)** Because the triangular lattice is bipartite.
- [ ] **(D)** Because the exchange parameter $J$ becomes complex.

In [44]:
ws.render_section('q22_35')

**Q35 · easy** · 💡

In a 2D square lattice of $N = 100$ spins ($10 \times 10$) with periodic boundary conditions, what is the total number of nearest-neighbor bonds $N_{\text{bonds}}$? (Recall coordination number $z = 4$).

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Each site has $z = 4$ neighbors, and each bond connects 2 sites: $N_{\text{bonds}} = z N / 2$.

</div>
</details>

_Numeric answer required._

In [45]:
ws.render_section('q22_36')

**Q36 · easy** · 💡

For the same $N = 100$ square lattice with $J = 1.0\text{ meV}$ and $h = 0$, what is the total ground state energy $E_{\text{ground}}$ in meV when all spins are fully aligned ($s_i = +1$ for all $i$)?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Multiply $-J$ by the number of bonds.

</div>
</details>

_Numeric answer required (in meV)._

In [46]:
ws.render_section('q22_37')

**Q37 · medium** · 💡

Starting from the fully aligned ground state ($E = -200\text{ meV}$), flipping a single spin $s_k \to -1$ in the bulk breaks $z = 4$ bonds. What is the energy change $\Delta E$ in meV?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Each of the 4 bonds changes from $-J$ to $+J$, costing $2J$ per bond: $\Delta E = 2 z J$.

</div>
</details>

_Numeric answer required (in meV)._

In [47]:
ws.render_section('q22_38')

**Q38 · medium** · 💡

How is the magnetic susceptibility $\chi_T$ related to the fluctuations of the total magnetization $M$ in the canonical ensemble at $h = 0$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Differentiate $\langle M \rangle = \frac{1}{N}\frac{\partial \ln Z}{\partial(\beta h)}$ with respect to $h$.

</div>
</details>

- [ ] **(A)** $\chi_T = \frac{N}{k_B T} \left( \langle M^2 \rangle - \langle M \rangle^2 \right)$
- [ ] **(B)** $\chi_T = \langle M \rangle^2$
- [ ] **(C)** $\chi_T = \frac{1}{N} \sum s_i$
- [ ] **(D)** $\chi_T = -k_B T \ln M$

In [48]:
ws.render_section('q22_39')

**Q39 · hard** · 💡

At the critical point $T = T_c$, how does the peak value of the magnetic susceptibility $\chi_{\max}(L)$ scale with linear lattice size $L$ according to finite-size scaling?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall the finite-size scaling relation for susceptibility $\chi(L) = L^{\gamma/\nu} \tilde{\chi}((T-T_c)L^{1/\nu})$.

</div>
</details>

- [ ] **(A)** $\chi_{\max}(L) \propto L^{\gamma / \nu} = L^{7/4} = L^{1.75}$
- [ ] **(B)** $\chi_{\max}(L) \propto L^0 = \text{constant}$
- [ ] **(C)** $\chi_{\max}(L) \propto e^{-L}$
- [ ] **(D)** $\chi_{\max}(L) \propto L^{-1}$

In [49]:
ws.render_section('q22_40')

**Q40 · medium** · 💡

In an experiment or simulation measuring the equal-time structure factor $S(k, t) = \langle |\hat{s}(\mathbf{k}, t)|^2 \rangle$ during coarsening, what universal scaling hypothesis is observed?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Think about self-similarity: the domain morphology at time $t_2$ looks identical to that at $t_1$ upon rescaling space by $L(t_2)/L(t_1)$.

</div>
</details>

- [ ] **(A)** Dynamical scaling hypothesis: $S(k, t) = L(t)^d F(k L(t))$, where $L(t)$ is the characteristic domain size and $F(x)$ is a time-independent master curve.
- [ ] **(B)** $S(k, t)$ is strictly constant in both $k$ and $t$.
- [ ] **(C)** $S(k, t)$ diverges to infinity at all wavevectors $k$.
- [ ] **(D)** $S(k, t) = e^{-k t}$

In [50]:
ws.render_section('q22_41')

**Q41 · hard** · 💡

What is Porod's law for the high-wavevector tail of the structure factor $S(k, t)$ in a system with sharp domain interfaces in $d$ dimensions?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

In 3D, $S(k) \propto k^{-4}$; in 2D, $S(k) \propto k^{-3}$.

</div>
</details>

- [ ] **(A)** $S(k, t) \propto k^{-(d+1)}$ as $k \to \infty$
- [ ] **(B)** $S(k, t) \propto e^{-k^2}$
- [ ] **(C)** $S(k, t) \propto k^2$
- [ ] **(D)** $S(k, t) \propto \ln(k)$

In [51]:
ws.render_section('q22_42')

**Q42 · medium** · 💡

In a 2D Ising simulation at $T_c$ with $L = 64$ using local Metropolis ($z = 2.17$), the autocorrelation time is $\tau_0 = 10,000\text{ MCS}$. If the lattice size is doubled to $L = 128$, calculate the new autocorrelation time $\tau = \tau_0 \times (128 / 64)^{2.17} = 10,000 \times 2^{2.17}$ in MCS to the nearest 100 MCS.

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Evaluate $2^{2.17} \approx 4.500$. Multiply by 10,000.

</div>
</details>

_Numeric answer required._

In [ ]:
ws.show_record()

In [ ]:
ws.finalize()